# Fase 1 — Ingestão da camada Bronze

Objetivo: carregar o `dataset.json` exatamente como ele chegou, sem corrigir
nenhum valor, e salvar como tabela Delta. A Bronze existe pra dar
rastreabilidade — se algo der errado mais na frente, sempre dá pra voltar
aqui e conferir o dado original, sem depender do GitHub de novo.

In [0]:
spark.sql("USE CATALOG fauna")
spark.sql("USE SCHEMA monitoramento")

caminho_dataset = "/Volumes/fauna/monitoramento/raw/dataset.json"
print(f"Lendo de: {caminho_dataset}")

In [0]:
df_bronze_raw = spark.read.option("multiLine", True).json(caminho_dataset)

print(f"Registros lidos: {df_bronze_raw.count()}")
df_bronze_raw.printSchema()

## Conferência rápida (contagem e amostra)

In [0]:
display(df_bronze_raw.limit(10))

In [0]:
total_esperado = 5000
total_lido = df_bronze_raw.count()

if total_lido == total_esperado:
    print(f"✅ Contagem bate: {total_lido} registros.")
else:
    print(f"⚠️ Contagem diferente: lidos {total_lido}, esperado {total_esperado}.")

## Metadados de ingestão

In [0]:
from pyspark.sql import functions as F

df_bronze = (
    df_bronze_raw
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

df_bronze.select("_ingestion_timestamp", "_source_file").show(5, truncate=False)

## Gravar tabela Bronze

In [0]:
TABELA_BRONZE = "fauna.monitoramento.bronze_registros"

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_BRONZE)
)

print(f"✅ Tabela criada: {TABELA_BRONZE}")

## Validação final da camada Bronze

In [0]:
df_check = spark.table(TABELA_BRONZE)

print(f"Total de linhas: {df_check.count()}")
print(f"Total de colunas: {len(df_check.columns)}")

print("\nCâmeras distintas:")
df_check.select("id_camera").distinct().orderBy("id_camera").show(20, truncate=False)

print("Grupos faunísticos distintos:")
df_check.select("grupo").distinct().show(truncate=False)

## Resumo e decisões da Fase 1 — Ingestão Bronze

Esta fase teve um único objetivo: trazer o `dataset.json` para dentro do
Databricks **exatamente como ele chegou**, sem corrigir nenhum valor. Isso é
o princípio central de uma camada Bronze — ela existe para dar
**rastreabilidade**: se qualquer coisa der errado nas camadas seguintes (Silver,
Gold), sempre dá pra voltar aqui e conferir o dado original, sem depender de
baixar o arquivo do GitHub de novo.

**Decisões tomadas e por quê:**

- **`multiLine=True` na leitura do JSON.** O `dataset.json` é um único array
  (`[ {...}, {...} ]`), não um arquivo "JSON Lines" (um objeto por linha). Sem
  essa opção, o Spark tentaria interpretar cada linha isoladamente e falharia
  ou leria o arquivo errado.

- **Nenhum schema definido manualmente.** Deixamos o Spark inferir os tipos a
  partir do dado real. Isso foi proposital: o schema inferido revelou que
  `temperatura_c` e `umidade_pct` vieram como `string` (por causa das unidades
  embutidas, tipo `"19.3 °C"`), e `data_hora_inicio` também como `string`.
  Corrigir isso é trabalho da Silver — aqui só precisávamos *ver* o problema,
  não resolvê-lo.

- **Metadados de ingestão (`_ingestion_timestamp`, `_source_file`).** Toda
  tabela Bronze deveria registrar de onde veio o dado e quando entrou no
  pipeline — essencial quando o mesmo pipeline roda repetidamente e é preciso
  saber qual execução trouxe qual registro.

- **`_metadata.file_path` em vez de `input_file_name()`.** Tentamos primeiro a
  função clássica do Spark (`input_file_name()`) e o Unity Catalog bloqueou,
  com o erro `UC_COMMAND_NOT_SUPPORTED` — essa função legada não é compatível
  com os recursos de governança do UC (linhagem de dados, segurança em nível
  de linha/coluna). A alternativa recomendada, e que usamos, é a coluna oculta
  `_metadata`, que o Spark anexa automaticamente a qualquer leitura de arquivo
  e é compatível com Unity Catalog.

- **`mode("overwrite")` + `overwriteSchema=true` na gravação.** Isso torna o
  notebook seguro para rodar de novo do zero (ex.: depois de uma correção) sem
  duplicar dados nem travar por causa de mudança de schema entre execuções.

**Resultado:** tabela `fauna.monitoramento.bronze_registros`, 5.000 registros,
12 câmeras (CAM01–CAM12) e 3 grupos faunísticos (Mamífero, Ave, Anfíbio)
confirmados na validação — batendo com o que o enunciado do checkpoint descreve.

**Próximo passo (Fase 2 — Silver):** tratar os tipos de `temperatura_c`,
`umidade_pct` e `data_hora_inicio`, e investigar duplicidades/valores fora do
esperado.